In [3]:
import requests

months = [
    ("2026-01-01", "2026-01-31"),
    ("2026-02-01", "2026-02-28"),
    ("2026-03-01", "2026-03-31")
]

all_earthquakes = []

for start, end in months:
    url = (
        f"https://earthquake.usgs.gov/fdsnws/event/1/query"
        f"?format=geojson&starttime={start}&endtime={end}"
    )

    response = requests.get(url)
    data = response.json()

    all_earthquakes.extend(data["features"])

print("Total earthquakes:", len(all_earthquakes))

Total earthquakes: 33360


In [4]:
import requests

# Month ranges for Q1 2026
month_ranges = [
    ("2026-01-01", "2026-01-31"),
    ("2026-02-01", "2026-02-28"),
    ("2026-03-01", "2026-03-31")
]

base_url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

all_features = []

for start_date, end_date in month_ranges:

    params = {
        "format": "geojson",
        "starttime": start_date,
        "endtime": end_date
    }

    response = requests.get(base_url, params=params)

    if response.status_code == 200:

        data = response.json()

        features = data.get("features", [])

        print(f"{start_date} to {end_date}: {len(features)} records")

        all_features.extend(features)

    else:
        print(f"Failed for {start_date} to {end_date}")

print("\nTotal earthquake records:", len(all_features))

2026-01-01 to 2026-01-31: 11886 records
2026-02-01 to 2026-02-28: 9777 records
2026-03-01 to 2026-03-31: 11697 records

Total earthquake records: 33360


In [5]:
records = []

for feature in all_features:

    properties = feature.get("properties", {})
    geometry = feature.get("geometry", {})
    coordinates = geometry.get("coordinates", [None, None, None])

    record = {
        "event_id": feature.get("id"),

        "magnitude": properties.get("mag"),
        "place": properties.get("place"),

        "time_epoch_ms": properties.get("time"),
        "updated_epoch_ms": properties.get("updated"),

        "felt": properties.get("felt"),
        "cdi": properties.get("cdi"),
        "mmi": properties.get("mmi"),
        "alert": properties.get("alert"),

        "status": properties.get("status"),
        "tsunami": properties.get("tsunami"),
        "significance": properties.get("sig"),

        "network": properties.get("net"),
        "num_stations": properties.get("nst"),

        "distance_deg": properties.get("dmin"),
        "rms": properties.get("rms"),
        "azimuthal_gap": properties.get("gap"),

        "magnitude_type": properties.get("magType"),
        "event_type": properties.get("type"),

        "longitude": coordinates[0],
        "latitude": coordinates[1],
        "depth_km": coordinates[2]
    }

    records.append(record)

In [6]:
import pandas as pd

df = pd.DataFrame(records)

In [7]:
print(df.head())

       event_id  magnitude                                  place  \
0  ak2026cdbrva        1.9             97 km N of Yakutat, Alaska   
1  ak2026cdbrru        1.9            87 km NW of Yakutat, Alaska   
2    nn00910214        2.4              51 km NE of Valmy, Nevada   
3  ak2026cdbppu        2.0            118 km N of Yakutat, Alaska   
4    us6000s90t        4.3  212 km SE of Pondaguitan, Philippines   

   time_epoch_ms  updated_epoch_ms  felt  cdi  mmi alert    status  ...  \
0  1769817419378     1770103517581   NaN  NaN  NaN  None  reviewed  ...   
1  1769817409401     1770103501571   NaN  NaN  NaN  None  reviewed  ...   
2  1769817287970     1769823324125   NaN  NaN  NaN  None  reviewed  ...   
3  1769817266158     1770102832524   NaN  NaN  NaN  None  reviewed  ...   
4  1769817245821     1776977004040   NaN  NaN  NaN  None  reviewed  ...   

   network  num_stations distance_deg     rms  azimuthal_gap  magnitude_type  \
0       ak            16        0.500  1.0000         

In [8]:
print(df.shape)

(33360, 22)


In [9]:
df["time"] = pd.to_datetime(
    df["time_epoch_ms"],
    unit="ms",
    utc=True
)

df["updated_time"] = pd.to_datetime(
    df["updated_epoch_ms"],
    unit="ms",
    utc=True
)

In [10]:
print(df[["time", "updated_time"]].head())

                              time                     updated_time
0 2026-01-30 23:56:59.378000+00:00 2026-02-03 07:25:17.581000+00:00
1 2026-01-30 23:56:49.401000+00:00 2026-02-03 07:25:01.571000+00:00
2 2026-01-30 23:54:47.970000+00:00 2026-01-31 01:35:24.125000+00:00
3 2026-01-30 23:54:26.158000+00:00 2026-02-03 07:13:52.524000+00:00
4 2026-01-30 23:54:05.821000+00:00 2026-04-23 20:43:24.040000+00:00


In [11]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33360 entries, 0 to 33359
Data columns (total 24 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   event_id          33360 non-null  object             
 1   magnitude         33348 non-null  float64            
 2   place             33360 non-null  object             
 3   time_epoch_ms     33360 non-null  int64              
 4   updated_epoch_ms  33360 non-null  int64              
 5   felt              1688 non-null   float64            
 6   cdi               1688 non-null   float64            
 7   mmi               694 non-null    float64            
 8   alert             199 non-null    object             
 9   status            33360 non-null  object             
 10  tsunami           33360 non-null  int64              
 11  significance      33360 non-null  int64              
 12  network           33360 non-null  object             
 13  n

In [12]:
print(df.describe())

          magnitude  time_epoch_ms  updated_epoch_ms         felt  \
count  33348.000000   3.336000e+04      3.336000e+04  1688.000000   
mean       1.752401   1.771074e+12      1.773177e+12    38.616114   
std        1.264557   2.270266e+09      3.680454e+09   315.133557   
min       -1.770000   1.767226e+12      1.767226e+12     0.000000   
25%        0.920000   1.769052e+12      1.770151e+12     1.000000   
50%        1.600000   1.771017e+12      1.772795e+12     2.000000   
75%        2.200000   1.773081e+12      1.775154e+12     5.000000   
max        7.500000   1.774915e+12      1.781795e+12  8212.000000   

               cdi         mmi       tsunami  significance  num_stations  \
count  1688.000000  694.000000  33360.000000  33360.000000  33360.000000   
mean      2.010012    3.043958      0.000989     72.046703     26.602878   
std       1.399860    1.381323      0.031437     96.713209     24.206553   
min       0.000000    0.000000      0.000000      0.000000      0.000000  

In [13]:
print(df.isnull().sum())

event_id                0
magnitude              12
place                   0
time_epoch_ms           0
updated_epoch_ms        0
felt                31672
cdi                 31672
mmi                 32666
alert               33161
status                  0
tsunami                 0
significance            0
network                 0
num_stations            0
distance_deg           10
rms                     0
azimuthal_gap           1
magnitude_type         12
event_type              0
longitude               0
latitude                0
depth_km                0
time                    0
updated_time            0
dtype: int64


In [14]:
print(df.nunique())

event_id            33360
magnitude             620
place               14416
time_epoch_ms       33360
updated_epoch_ms    27074
felt                  128
cdi                    45
mmi                   595
alert                   2
status                  2
tsunami                 2
significance          417
network                15
num_stations          229
distance_deg        13026
rms                   805
azimuthal_gap         346
magnitude_type          8
event_type              9
longitude           24071
latitude            22035
depth_km             8628
time                33360
updated_time        27074
dtype: int64


In [15]:
# ---------------------------
# Handle Null Values
# ---------------------------

df["magnitude"] = df["magnitude"].fillna(0.0)

df["felt"] = df["felt"].fillna("none")
df["cdi"] = df["cdi"].fillna("none")
df["mmi"] = df["mmi"].fillna("none")
df["alert"] = df["alert"].fillna("none")

df["distance_deg"] = df["distance_deg"].fillna(0.0)

df["rms"] = df["rms"].fillna(0.0)

df["azimuthal_gap"] = df["azimuthal_gap"].fillna(0.0)

In [16]:
# tsunami should be 0 or 1
df["tsunami"] = df["tsunami"].astype(int)

# significance as integer
df["significance"] = df["significance"].astype(int)

# safe integer conversion
df["num_stations"] = (
    pd.to_numeric(df["num_stations"], errors="coerce")
      .fillna(0)
      .astype(int)
)

# float columns
df["magnitude"] = df["magnitude"].astype(float)
df["depth_km"] = df["depth_km"].astype(float)
df["latitude"] = df["latitude"].astype(float)
df["longitude"] = df["longitude"].astype(float)

In [17]:
bins = [-float("inf"), 2.0, 4.0, 5.0, 6.0, 7.0, float("inf")]

labels = [
    "Micro",
    "Minor",
    "Light",
    "Moderate",
    "Strong",
    "Major"
]

df["magnitude_category"] = pd.cut(
    df["magnitude"],
    bins=bins,
    labels=labels,
    right=False
)

In [18]:
print("Duplicate event_ids:",
      df["event_id"].duplicated().sum())

Duplicate event_ids: 0


In [19]:
df = df.drop_duplicates(
    subset="event_id",
    keep="first"
)

In [20]:
print("Rows after deduplication:", len(df))

Rows after deduplication: 33360


In [21]:
print(df.info())

print(df.isnull().sum())

print(df["magnitude_category"].value_counts())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33360 entries, 0 to 33359
Data columns (total 25 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   event_id            33360 non-null  object             
 1   magnitude           33360 non-null  float64            
 2   place               33360 non-null  object             
 3   time_epoch_ms       33360 non-null  int64              
 4   updated_epoch_ms    33360 non-null  int64              
 5   felt                33360 non-null  object             
 6   cdi                 33360 non-null  object             
 7   mmi                 33360 non-null  object             
 8   alert               33360 non-null  object             
 9   status              33360 non-null  object             
 10  tsunami             33360 non-null  int32              
 11  significance        33360 non-null  int32              
 12  network             33360 non-nu

In [22]:
df["magnitude_type"] = df["magnitude_type"].fillna("unknown")

In [23]:
print(df.isnull().sum())

event_id              0
magnitude             0
place                 0
time_epoch_ms         0
updated_epoch_ms      0
felt                  0
cdi                   0
mmi                   0
alert                 0
status                0
tsunami               0
significance          0
network               0
num_stations          0
distance_deg          0
rms                   0
azimuthal_gap         0
magnitude_type        0
event_type            0
longitude             0
latitude              0
depth_km              0
time                  0
updated_time          0
magnitude_category    0
dtype: int64


In [24]:
df.to_csv(
    "earthquakes_q1_2026_clean.csv",
    index=False
)

print("CSV file saved successfully.")

CSV file saved successfully.


In [25]:
import pandas as pd

df = pd.read_csv("earthquakes_q1_2026_clean.csv")
print(df.shape)
print(df.columns.tolist())

(33360, 25)
['event_id', 'magnitude', 'place', 'time_epoch_ms', 'updated_epoch_ms', 'felt', 'cdi', 'mmi', 'alert', 'status', 'tsunami', 'significance', 'network', 'num_stations', 'distance_deg', 'rms', 'azimuthal_gap', 'magnitude_type', 'event_type', 'longitude', 'latitude', 'depth_km', 'time', 'updated_time', 'magnitude_category']


In [1]:
pip install mysql-connector-python pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
df.to_csv("earthquakes_q1_2026_clean.csv", index=False)

In [27]:
print(df.shape)

(33360, 25)


In [33]:
import pandas as pd
import mysql.connector

df = pd.read_csv("earthquakes_q1_2026_clean.csv")

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="SQLlearner",
    database="seismic_db"
)

cursor = conn.cursor()

In [34]:
insert_query = """
INSERT IGNORE INTO earthquakes(
event_id,
magnitude,
place,
time_epoch_ms,
updated_epoch_ms,
felt,
cdi,
mmi,
alert,
status,
tsunami,
significance,
network,
num_stations,
distance_deg,
rms,
azimuthal_gap,
magnitude_type,
event_type,
longitude,
latitude,
depth_km,
time,
updated_time,
magnitude_category
)
VALUES (
%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,
%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,
%s,%s,%s,%s,%s
)
"""

In [35]:
records = [tuple(row) for row in df.itertuples(index=False)]

In [36]:
print(records[0])

('ak2026cdbrva', 1.9, '97 km N of Yakutat, Alaska', 1769817419378, 1770103517581, 'none', 'none', 'none', 'none', 'reviewed', 0, 56, 'ak', 16, 0.5, 1.0, 56.0, 'ml', 'earthquake', -139.495, 60.415, 3.4, '2026-01-30 23:56:59.378000+00:00', '2026-02-03 07:25:17.581000+00:00', 'Micro')


In [37]:
cursor.executemany(insert_query, records)

conn.commit()

print("Rows inserted:", cursor.rowcount)

Rows inserted: 33360


In [38]:
cursor.close()
conn.close()

print("Loading Completed")

Loading Completed
